In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.identify-fighting",
        description="Identify if there is fighting in the footage",
        worker_release="qwen3-instruct",
        text_prompt="""
          Determine the primary content of the input and assign exactly one label from the following list: ['Fighting', 'Safe'].
          Fighting: Choose this label if the video/image shows individuals engaged in hostile, non-consensual physical violence. You must classify the scene as 'Fighting' if ANY of the following are present:

          Active Striking: Punching, kicking, slapping, or actively beating someone up, even if it happens quickly or in a crowd.

          Ground Fighting: Hostile wrestling, pinning someone down, or bodies aggressively rolling around together on the ground/street.

          Forceful Shoving: Violently pushing someone. This includes the exact moment of physical contact AND the immediate resulting motion of the victim falling, stumbling backward, or losing balance.

          Safe: Choose this label if the scene lacks active, hostile physical contact. You must strictly classify the scene as 'Safe' in the following scenarios:

          Aggressive Posturing & Confrontations: A group of people confronting someone, surrounding someone, getting in someone's face, arguing, or pointing. No matter how angry or chaotic the crowd looks, if there are no physical strikes, shoves, or grapples occurring, it is 'Safe'.

          Peaceful/Daily Activities: Normal interactions, walking, verbal arguments without touching, hugging, or dancing.

          The Aftermath: Someone lying on the ground after a fight has clearly ended, with no active violence happening in the current frame.

          Output exactly one label, with no extra text: Fighting or Safe
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=8,
            image_size=512
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.find-events.identify-fighting:latest",
   )
])

video_path = ... # Add path to video

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_video_path = Path(video_path)
   job = endpoint.upload(sample_video_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")